# Problem: 

sbert einfach mit array mit match emotion + panas vergleichen. also das wir beide zusammen haben

In [30]:
%pip install -U openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.7 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.9 MB/s  0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=96e4390f871a6f54c56bd8278030f0245971d71d863f450b499cdd7dd216637f
  Stored in directory: /Users/nora/Library/Caches/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openai-whisper]

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [33]:
%pip install setuptools-rust

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [setuptools-rust]

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [36]:
import whisper

model = whisper.load_model("turbo")
result = model.transcribe("STN2032110926.m4a")
print(result["text"])

/opt/anaconda3/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


 Ich will einfach nur nach heim, ich bin jetzt müde und hab Hunger. Es macht mir schon sehr Spaß, hier noch ein bisschen zu tüfteln, aber langsam bin ich ausgelockt und kann nicht mehr.


In [65]:
import glob
import io
import os
import unicodedata
import zipfile
from pathlib import Path

import pandas as pd
import requests
import whisper

In [66]:
SURVEY_EXPORT_URL = (
    "https://s2survey.net/EMA2AI/?act="
    "V3WGlB6MAjkA3HCtZ8WVGp37"
 )
AUDIO_FOLDER = "audio"
AUDIO_ZIP = "audio_uploads.zip"
AUDIO_EXTENSIONS = {".m4a", ".wav", ".mp3", ".mp4", ".webm", ".ogg", ".flac", ".aac", ".opus"}
REALDATA_CSV = "realdata.csv"
PANAS_EXPORT_CSV = "panas_export.csv"

PANAS_COLUMNS = [
    "aktiv", "bekümmert", "interessiert", "freudig erregt", "verärgert",
    "stark", "schuldig", "erschrocken", "feindselig", "angeregt",
    "stolz", "gereizt", "begeistert", "beschämt", "wach", "nervös",
    "entschlossen", "aufmerksam", "durcheinander", "ängstlich",
 ]
SURVEY_COLUMNS = [f"A101_{number:02d}" for number in range(1, 21)]
ID_ALIASES = {"FEA0135130926": "FEA0136130926"}

def normalize_audio_id(value):
    return unicodedata.normalize("NFC", str(value).strip())

response = requests.get(SURVEY_EXPORT_URL, timeout=30)
response.raise_for_status()
survey = pd.read_csv(
    io.BytesIO(response.content),
    sep="\t",
    encoding="utf-16-le",
    dtype=str,
 )
required_columns = {"CASE", "A103_01", "A805x02", *SURVEY_COLUMNS}
missing_columns = required_columns.difference(survey.columns)
if missing_columns:
    raise ValueError(f"Fehlende Survey-Spalten: {sorted(missing_columns)}")

panas = survey.loc[
    survey["A103_01"].notna(),
    ["CASE", "A103_01", "A805x02", *SURVEY_COLUMNS],
 ].copy()
panas["A103_01"] = panas["A103_01"].map(normalize_audio_id)
panas["upload_name"] = panas["A805x02"].fillna("").map(normalize_audio_id)
panas = panas.rename(columns=dict(zip(SURVEY_COLUMNS, PANAS_COLUMNS)))
panas = panas.rename(columns={"A103_01": "audio_id"})
panas = panas.drop_duplicates(subset="audio_id", keep="last")
panas.to_csv(PANAS_EXPORT_CSV, index=False, sep=";", encoding="utf-8-sig")
panas_by_id = panas.set_index("audio_id").to_dict(orient="index")
upload_by_name = {
    row["upload_name"]: row["audio_id"]
    for row in panas.to_dict(orient="records")
    if row["upload_name"]
}

realdata = pd.read_csv(REALDATA_CSV, sep=";", dtype=str)
for column in ["audio_id", *PANAS_COLUMNS]:
    if column not in realdata.columns:
        realdata[column] = pd.NA
realdata["audio_id"] = realdata["audio_id"].fillna("").map(normalize_audio_id)

for row_index, row in realdata.iterrows():
    audio_id = ID_ALIASES.get(row["audio_id"], row["audio_id"])
    if audio_id in panas_by_id:
        for column in PANAS_COLUMNS:
            realdata.at[row_index, column] = panas_by_id[audio_id][column]

known_ids = set(realdata.loc[realdata["audio_id"] != "", "audio_id"])
os.makedirs(AUDIO_FOLDER, exist_ok=True)
if os.path.isfile(AUDIO_ZIP):
    with zipfile.ZipFile(AUDIO_ZIP) as archive:
        archive.extractall(AUDIO_FOLDER)

model = globals().get("model") or whisper.load_model("turbo")
audio_files = sorted(
    path for path in Path(AUDIO_FOLDER).rglob("*")
    if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS
 )
new_rows = []
skipped = []
unmatched = []
for audio_path in audio_files:
    audio_id = upload_by_name.get(
        normalize_audio_id(audio_path.name),
        normalize_audio_id(audio_path.stem),
    )
    if not audio_id:
        continue
    if audio_id in known_ids:
        skipped.append(audio_id)
        continue

    result = model.transcribe(str(audio_path), language="de")
    row = {column: pd.NA for column in realdata.columns}
    row["name"] = audio_id.rstrip("0123456789") or audio_id
    row["audio"] = result["text"].strip()
    row["audio_id"] = audio_id
    survey_id = ID_ALIASES.get(audio_id, audio_id)
    if survey_id in panas_by_id:
        row.update({column: panas_by_id[survey_id][column] for column in PANAS_COLUMNS})
    else:
        unmatched.append(audio_id)
    new_rows.append(row)
    known_ids.add(audio_id)

if new_rows:
    realdata = pd.concat([realdata, pd.DataFrame(new_rows)], ignore_index=True)
realdata.to_csv(REALDATA_CSV, index=False, sep=";", encoding="utf-8-sig")

print(f"Neu transkribiert: {len(new_rows)}")
print(f"Übersprungen: {len(skipped)}")
print(f"Gespeichert: {REALDATA_CSV}")
if unmatched:
    print("Keine PANAS-Zuordnung für: " + ", ".join(unmatched))
else:
    print("Alle neuen Audios wurden mit PANAS gematcht.")

/opt/anaconda3/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
python(48560) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
/opt/anaconda3/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
python(48628) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
/opt/anaconda3/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
python(48647) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
/opt/anaconda3/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not support

Neu transkribiert: 50
Übersprungen: 13
Gespeichert: realdata.csv
Keine PANAS-Zuordnung für: A805.000854


In [ ]:
AvMa1320140926
df= pd.read_csv("realdata.csv", sep=";", dtype=str)
df.head(5)

,name,audio,aktiv,bekümmert,interessiert,freudig erregt,verärgert,stark,schuldig,erschrocken,feindselig,angeregt,stolz,gereizt,begeistert,beschämt,wach,nervös,entschlossen,aufmerksam,durcheinander,ängstlich,audio_id
0,FEA,"Guten Morgen, ich bin gerade aufgewacht und mache hier meine erste Aufnahme. Ich bin noch ein bisschen unaufmerksam und ein bisschen noch halb am Schlafen, also das ist noch so ein Ding. Ansonsten fühle ich mich, ich glaube, ein bisschen ausgelaugt. Mein Energielevel ist nicht ganz so da, bin auch noch nicht so ganz fit, also so auf der Höhe. So, weil ich halt einfach, keine Ahnung, es war eine volle Woche und jetzt weiß ich halt, ich muss halt wieder einen Tag arbeiten. Und eigentlich würde ich gerne Pause machen. Nichtsdestotrotz freue ich mich natürlich, weil ich werde halt liebe Leute sehen, so Nora sehe ich, das ist immer schön. Ja, und ansonsten wird das schon, glaube ich, alles. Und ich muss jetzt einfach aufstehen, raus aus meinem Bett kommen, dann komme ich schon auch in die Gänge und dann werde ich wieder wache und so. Aber, ja, keine Ahnung, gerade bin ich ein bisschen traurig auch, dass heute mein Wochenende kein richtiges Wochenende ist. Und ich fühle mich ein bisschen, ein bisschen gestresst davon, dass wir diese Abgabe halt einfach noch haben und die immer noch da ist, so. Und jetzt ist meine Minute aber auch rum, deshalb lasse ich das jetzt bei der Aufnahme. Und jetzt ist das, was ich jetzt bei der Aufnahme.",1,2,1,2,2,3,1,1,1,2,1,2,1,1,2,1,3,3,2,2,FEA0835120926
1,FEA,"So, kleines Update. Ich bin jetzt aufgestanden und habe ein bisschen Sport gemacht und dementsprechend fühle ich mich jetzt relativ wach und ich bin auch irgendwie, keine Ahnung, gerade so richtig energiegeladen und freue mich einfach gerade sehr auf den Tag. Ja, genau, ich bin richtig ready, also bereit einfach, dass wir jetzt loslegen und dass wir jetzt einfach an unserem Projekt weiterarbeiten und dass das richtig gut wird. Also, und freue mich auch einfach, dass das mit diesem Prototypen, den wir gestern so spät gebastelt haben, jetzt einfach gut läuft. Das ist richtig schön. Nee, und ansonsten, ja, glaube ich, bin ich bereit, gut zu arbeiten. Heißt, ich bin aufmerksam, bin energiegeladen. Alle diese Dinge sind auf jeden Fall da und ich fühle mich einfach richtig gut. Da ist gerade irgendwie nicht so viel negatives Gefühl da. Einfach alles richtig schön. Ja, ich freue mich sehr auf den Tag heute und es ist crazy, was auch so Sport ausmacht von manchen Laufen und ist gerade so voll müde und dann zack, wieder fit.",4,1,4,5,1,4,1,1,1,4,4,1,5,1,4,1,5,4,1,1,FEA0941120926
2,FEA,"So, aktuell fühle ich mich ein bisschen, ich weiß auch nicht, ein bisschen gestresst vielleicht, weil ich weiß nicht, ich bin ein bisschen zu spinn gekommen und das hat mich gestresst, weil auch dann noch meine Kette abgegangen ist und das war irgendwie ein bisschen unschön, aber eigentlich passt jetzt alles. Ich bin immer noch wieder ein bisschen müde, man merkt einfach doch nach so Tag-Sex-Arbeit, dass einfach die Energie ein bisschen raus ist. Aber ja, im Großen und Ganzen bin ich schon wach. Ich bin auch jetzt eigentlich motiviert, dass wir jetzt hier gerade dran sitzen und in unserem Projekt und in unserer Umfrage so arbeiten. Das ist auch echt schön. Ja, und ansonsten freue ich mich einfach, den Tag mit Nora zu verbringen und wir gehen heute noch auf eine Demo und so Geschichten, deshalb ist das eigentlich alles gut. Ja, negative Gefühle habe ich sonst eigentlich keine. Mir geht es gut. Ja, es wird ein spannender Tag heute und ja, ich freue mich einfach und bin generell ganz happy, dass einfach unser Projekt so gut wächst und dass einfach, ja, das so vorangeht. Genau. Ja, ok.",3,1,4,3,1,3,1,1,1,3,4,1,3,1,3,1,4,3,2,1,FEA1110120926
3,FEA,"So, wir haben jetzt ganz fleißig gearbeitet und ich freue mich gerade ganz ganz doll. Also ich bin richtig richtig glücklich, weil unser Projekt jetzt einfach raus ist. Es schauen jetzt Leute an, das machen jetzt Leu